# Issue #208 — dynamic N×K and risk-conditioned parlay policy frontier

Contract BJ-UNIFIED-DYNAMIC-N-001 / v3. This notebook verifies the immutable
package, renders the frozen chronological decision, the action frontier, the
N×K performance surfaces, overlap/concentration diagnostics and the selection
receipt. Core generation, modeling, settlement and verification live in
`src/parlay_policy/`.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

def _find_root():
    candidate = Path.cwd()
    for _ in range(6):
        if (candidate / 'results' / 'issue-208').exists():
            return candidate
        candidate = candidate.parent
    return Path.cwd()

ROOT = _find_root()
sys.path.insert(0, str(ROOT / 'src'))

PACKAGE = ROOT / 'results' / 'issue-208'
selection = json.loads((PACKAGE / 'selection.json').read_text(encoding='utf-8'))
decision = json.loads((PACKAGE / 'decision.json').read_text(encoding='utf-8'))
handoff = json.loads((PACKAGE / 'handoff_209.json').read_text(encoding='utf-8'))
selection, decision['contract_version']

({'candidate': {'capital': 'NATIVE_2_YUAN',
   'k': 8,
   'lane': 'MARKET_WOR',
   'n': 7,
   'policy': 'FROZEN_N7_INCUMBENT',
   'reason': 'selection rule found no eligible learned row; freeze N7 incumbent'},
  'eligible': False,
  'frozen_candidate': {'capital': 'NATIVE_2_YUAN',
   'k': 8,
   'lane': 'MARKET_WOR',
   'n': 7,
   'policy': 'FROZEN_N7_INCUMBENT'},
  'matched_lift_by_fold': [{'fold': 'fold1',
    'frozen_incumbent_end': 99072.0,
    'learned_end': 99072.0,
    'matched_lift': 0.0,
    'test_split': 'CV_2022'},
   {'fold': 'fold2',
    'frozen_incumbent_end': 99312.88946088498,
    'learned_end': 99312.88946088498,
    'matched_lift': 0.0,
    'test_split': 'CV_2023'}],
  'median_matched_lift_yuan': 0.0,
  'positive_fold_count': 0,
  'rule1_valid_replay_no_ruin': True,
  'rule2_positive_median_lift': False,
  'rule3_positive_in_two_folds': False},
 'bj208-dynamic-nk-policy-v1')

## 1. Data census

171 development issues (CV_2021 58 / CV_2022 58 / CV_2023 55) from the frozen
#205 value matrix; 2024/2025/2026 are embargoed and never enter #208.

In [2]:
census = pd.read_csv(PACKAGE / 'census.csv')
census.groupby('split').agg(issues=('issue', 'nunique'), mean_M=('M', 'mean'), labels=('labels_available', 'sum'))

,issues,mean_M,labels
split,,,
CV_2021,58,321.103448,58
CV_2022,58,285.844828,58
CV_2023,55,283.545455,55
LOCKED_TEST,75,226.240000,0


## 2. Action frontier (counterfactual label surface)

Per-issue matched pre-tax capital lift versus the frozen N7/K8 incumbent for
all 168 action cells on the CV_2021 fit fold. Best cells come from the
LEARNED_MLP lane (the strongest non-promoted #207 research lane).

In [3]:
surface = pd.read_parquet(PACKAGE / 'label_surface_fit.parquet')
print('surface rows:', len(surface))
print('mean matched lift by lane:')
surface.groupby('lane')['matched_lift_mean'].agg(['mean', 'median', 'count']).round(2)

surface rows: 9744
mean matched lift by lane:


,mean,median,count
lane,,,
LEARNED_MLP,-1.29,-16.0,2436
MARKET_WOR,-8.56,-16.0,2436
MARKET_WR,-9.84,-16.0,2436
VALUE_WOR,-13.51,-16.0,2436


## 3. N×K performance surface (CV_2021 fit fold)

Median matched lift over lanes/capital, as a function of N and K. Negative
median lifts dominate: most action cells lose to the frozen incumbent on
matched pre-tax capital.

In [4]:
pivot = surface.pivot_table(index='n', columns='k', values='matched_lift_mean', aggfunc='median').round(1)
pivot

k,8,16,32
n,,,
4,0.0,-16.0,-37.8
5,0.0,-16.0,-48.0
6,0.0,-16.0,-48.0
7,0.0,-16.0,-48.0
8,0.0,-16.0,-48.0
9,0.0,-16.0,-48.0
10,0.0,-16.0,-48.0


## 4. Chronological learned policy (safety-gated)

Fold 1 (train CV_2021 → test CV_2022) and Fold 2 (train CV_2021+CV_2022 → test
CV_2023). The baseline-residual safety gate (conservative lift = prediction −
1×RMSE, plus coverage/OOD/fundability) passes on only 3 of 113 test issues;
everywhere else the policy falls back to the frozen incumbent. The gated
learned path therefore equals the frozen incumbent path on both folds.

In [5]:
for fold in ('fold1', 'fold2'):
    learned = pd.read_parquet(PACKAGE / f'path_learned_{fold}.parquet')
    incumbent = pd.read_parquet(PACKAGE / f'path_control_incumbent_n7k8_{fold}.parquet')
    print(f'{fold}: learned end={learned.post_issue_bankroll_yuan.iloc[-1]:,.2f}  '
          f'incumbent end={incumbent.post_issue_bankroll_yuan.iloc[-1]:,.2f}  '
          f'sources={learned.source.value_counts().to_dict()}')

fold1: learned end=99,072.00  incumbent end=99,072.00  sources={'FROZEN_INCUMBENT_207': 58}
fold2: learned end=99,312.89  incumbent end=99,312.89  sources={'FROZEN_INCUMBENT_207': 52, 'REPLAY': 3}


## 5. Controls / attribution matrix (CV_2023 test)

Ten-arm attribution matrix on the terminal fold. The frozen incumbent arm is
bound exactly to the #207 CONTROLLING K8 path. Cash/NO_BET is reported as a
safety reference, outside active-Edge attribution.

In [6]:
import glob
rows = []
for f in sorted(glob.glob(str(PACKAGE / 'path_control_*_fold2.parquet'))):
    name = Path(f).name.split('path_control_')[1].split('_fold2')[0]
    d = pd.read_parquet(f)
    rows.append({'control': name, 'ending_capital': d.post_issue_bankroll_yuan.iloc[-1],
                 'turnover': d.funded_stake_yuan.sum()})
controls = pd.DataFrame(rows).sort_values('ending_capital', ascending=False)
controls.round(2)

,control,ending_capital,turnover
0,capital_change_identical,101424.58,880.0
5,generator_change_n7k8,101424.58,880.0
4,fixed_n7_dynamic_k,101424.58,880.0
11,static_n6k8,100020.03,880.0
1,cash_no_bet,100000.00,0.0
3,dynamic_nk_fixed_generator,99818.56,880.0
2,dynamic_n_fixed_k8_lane,99818.56,880.0
9,static_n4k8,99818.56,880.0
10,static_n5k8,99496.82,880.0
6,incumbent_n7k8,99312.89,880.0


## 6. Learning diagnostics

Calibration of predicted vs realized best lift is monotone (higher predicted
lift → higher realized lift), so the Q model has signal. Action frequency
shows the ungated policy prefers short parlays (N=4/5) and the LEARNED_MLP
lane; the safety gate vetoes nearly all of them.

In [7]:
calib = pd.read_csv(PACKAGE / 'calibration_curve.csv')
calib.round(2)

,bin,predicted_best_lift,realized_best_lift,issues
0,0,7.87,63.58,24
1,1,17.83,167.36,23
2,2,25.67,110.67,23
3,3,39.46,161.21,23
4,4,80.68,827.34,23


In [8]:
freq = pd.read_csv(PACKAGE / 'action_frequency.csv')
freq.head(8)

,action_key,n,k,lane,capital,count,share
0,N4/K8/LEARNED_MLP/NATIVE_2_YUAN,4,8,LEARNED_MLP,NATIVE_2_YUAN,20,0.176991
1,N4/K8/MARKET_WR/NATIVE_2_YUAN,4,8,MARKET_WR,NATIVE_2_YUAN,19,0.168142
2,N4/K8/LEARNED_MLP/SIMPLE_ACTIVE,4,8,LEARNED_MLP,SIMPLE_ACTIVE,14,0.123894
3,N5/K8/LEARNED_MLP/NATIVE_2_YUAN,5,8,LEARNED_MLP,NATIVE_2_YUAN,8,0.070796
4,N5/K8/LEARNED_MLP/SIMPLE_ACTIVE,5,8,LEARNED_MLP,SIMPLE_ACTIVE,5,0.044248
5,N6/K8/MARKET_WR/NATIVE_2_YUAN,6,8,MARKET_WR,NATIVE_2_YUAN,5,0.044248
6,N5/K8/MARKET_WR/NATIVE_2_YUAN,5,8,MARKET_WR,NATIVE_2_YUAN,4,0.035398
7,N4/K16/LEARNED_MLP/SIMPLE_ACTIVE,4,16,LEARNED_MLP,SIMPLE_ACTIVE,4,0.035398


## 7. Selection receipt and #209 handoff

Lexicographic selection rule: no learned row passes rules 2/3 (median matched
lift = 0.0; positive folds = 0), so the frozen N7/K8/MARKET_WOR/native-¥2
incumbent is frozen for #209 and the full frontier is published. 2024 was
never opened.

In [9]:
print('eligible:', selection['eligible'])
print('median matched lift:', selection['median_matched_lift_yuan'])
print('positive folds:', selection['positive_fold_count'])
print('matched by fold:', selection['matched_lift_by_fold'])
print('candidate:', selection['candidate'])
print('handoff 2024_opened:', handoff['fit_boundary']['2024_opened'])

eligible: False
median matched lift: 0.0
positive folds: 0
matched by fold: [{'fold': 'fold1', 'frozen_incumbent_end': 99072.0, 'learned_end': 99072.0, 'matched_lift': 0.0, 'test_split': 'CV_2022'}, {'fold': 'fold2', 'frozen_incumbent_end': 99312.88946088498, 'learned_end': 99312.88946088498, 'matched_lift': 0.0, 'test_split': 'CV_2023'}]
candidate: {'capital': 'NATIVE_2_YUAN', 'k': 8, 'lane': 'MARKET_WOR', 'n': 7, 'policy': 'FROZEN_N7_INCUMBENT', 'reason': 'selection rule found no eligible learned row; freeze N7 incumbent'}
handoff 2024_opened: False


## 8. Package verification + tamper self-tests (REQ-E06)

Recompute manifest hashes, capital accounting, path continuity, embargoed
split absence and run the four tamper attacks. Every attack must be detected.

In [10]:
from parlay_policy.verify import verify_manifest, verify_capital_accounting, verify_no_embargoed_split, verify_no_2024_issue

manifest_json = json.loads((PACKAGE / 'SHA256SUMS.json').read_text(encoding='utf-8'))
hash_failures = verify_manifest(PACKAGE, manifest_json)
ledger = pd.read_parquet(PACKAGE / 'issue_capital_ledger.parquet')
acct_failures = verify_capital_accounting(ledger)
embargo_failures = verify_no_embargoed_split(ledger) + verify_no_2024_issue(ledger)
print('manifest failures:', hash_failures)
print('accounting failures:', acct_failures)
print('embargo/2024 failures:', embargo_failures)
print('package verification:', 'PASS' if not (hash_failures or acct_failures or embargo_failures) else 'FAIL')

manifest failures: []
accounting failures: []
embargo/2024 failures: []
package verification: PASS


In [11]:
tamper = json.loads((PACKAGE / 'tamper_results.json').read_text(encoding='utf-8'))
for result in tamper:
    print(f"  {result['attack']:28s} detected={result['detected']}")
assert all(r['detected'] for r in tamper), 'tamper suite must detect all attacks'
print('tamper self-tests: PASS (4/4)')

  mutate_settlement            detected=True
  self_consistent_ledger       detected=True
  forge_manifest               detected=True
  forge_source_trace           detected=True
tamper self-tests: PASS (4/4)


## 9. Boundary statement

- 2024 calendar year: embargoed (no 2024 issue appears in any #208 artifact).
- 2025/2026 LOCKED_TEST: embargoed.
- Selection: freeze the frozen N7 incumbent; publish the full frontier and
  failure regimes per the preregistered lexicographic rule.
- #209 receives one byte-identifiable handoff (`handoff_209.json`) with model
  class, feature schema, fitted weights config, preprocessing, safety
  threshold and inference code; 2024 was not opened.